# Classification with PyTorch V1.3 (oracle) — Response, deeper MLP, k-fold validation (k=5)

**Version:** V1.3 (k-fold validation — deeper MLP). **Role:** reference oracle for the V1.3 scratch
twin. Identical `ClfDeepMLP` architecture and hypers as `classification_pytorch_V1_1.ipynb`
(`FC(22→64)→BN→ReLU→Dropout(0.2)→FC(64→64)→BN→(+residual)→ReLU→Dropout(0.2)→FC(64→1)`, weighted BCE,
Adam+cosine+wd(1e-4), 100 epochs), copied verbatim and wrapped in a 5-fold `StratifiedKFold` loop over
the full 2,236-row cleaned set instead of one fixed 80/20 split — this is the single-split-vs-small-n
robustness check for V1.1's deeper MLP, pulled out into its own standalone version so it sits as a
sibling of V2.1/V3.1 rather than appended inside V1.1's own notebook. Weights → `./weights_V1_3/`.

In [1]:
import json, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs("weights_V1_3", exist_ok=True)
print("torch:", torch.__version__)

torch: 2.14.1+cu130


## 1. LOAD DATA (identical to V1.1)

In [2]:
df = pd.read_csv("marketing_campaign.csv", sep="\t")
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
ref_date = df["Dt_Customer"].max()
df["Customer_Tenure_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Age"] = 2014 - df["Year_Birth"]
df = df[~df["Year_Birth"].isin([1893, 1899, 1900])].copy()
df = df[df["Income"] != 666666].copy()  # drop $666,666 income outlier (same treatment as Year_Birth sentinels)
df["Marital_Status"] = df["Marital_Status"].replace({"Absurd": "Single", "YOLO": "Single", "Alone": "Single"})
print("cleaned:", df.shape, "| Response rate:", round(df["Response"].mean(), 4))

CLF_NUMERIC = [
    "AcceptedCmp5", "AcceptedCmp3", "MntWines", "MntMeatProducts",
    "NumCatalogPurchases", "Recency", "Customer_Tenure_Days", "MntGoldProds",
    "Income", "MntFruits",
    "AcceptedCmp1", "AcceptedCmp2", "NumWebPurchases", "MntSweetProducts",
    "MntFishProducts", "NumStorePurchases", "Age",
]
X_num = df[CLF_NUMERIC].copy()
X_cat = pd.get_dummies(df[["Marital_Status"]], dtype=float)
X_full = pd.concat([X_num, X_cat], axis=1)
FEATURE_COLS = list(X_full.columns)
y_full = df["Response"].astype(float).values
inc_idx = FEATURE_COLS.index("Income")
print(f"features: {len(CLF_NUMERIC)} numeric + {X_cat.shape[1]} dummies = {len(FEATURE_COLS)} dims")

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(np.asarray(X, dtype=np.float32))
        self.y = torch.from_numpy(np.asarray(y, dtype=np.float32)).view(-1, 1)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, i):
        return self.X[i], self.y[i]

cleaned: (2236, 28) | Response rate: 0.1494
features: 17 numeric + 5 dummies = 22 dims


## 2. MODEL: ClfDeepMLP (identical to V1.1)

In [3]:
class ClfDeepMLP(nn.Module):
    def __init__(self, in_dim=22, h=64, p=0.2):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, h)
        self.bn1 = nn.BatchNorm1d(h)
        self.fc2 = nn.Linear(h, h)
        self.bn2 = nn.BatchNorm1d(h)
        self.fc3 = nn.Linear(h, 1)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(p)
        for fc, nl in ((self.fc1, "relu"), (self.fc2, "relu"), (self.fc3, "linear")):
            nn.init.kaiming_normal_(fc.weight, nonlinearity=nl)
            nn.init.zeros_(fc.bias)
    def forward(self, x):
        d1 = self.drop(self.relu(self.bn1(self.fc1(x))))
        s = self.bn2(self.fc2(d1)) + d1  # residual around the 64->64 block
        d2 = self.drop(self.relu(s))
        return self.fc3(d2)  # logits

device = torch.device("cpu")
bce = nn.BCEWithLogitsLoss()

## K-fold training (k=5) — fresh ClfDeepMLP per fold, no checkpointing

Same convention as the V1.1 single split: each fold trains for a fixed 100 epochs (no best-epoch
checkpoint selection) and is evaluated on its own held-out fold at the end. Each fold re-fits its own
median-fill value and `StandardScaler` on that fold's training rows only, and trains a **fresh** model
(new init, new Adam/scheduler state) — no information crosses from one fold to another. Saved to
`weights_V1_3/classification_pytorch_kfold_metrics.json`.

In [4]:
K = 5
skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)
X_all = X_full.values
fold_rows = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_all, y_full), start=1):
    Xtr_f, Xva_f = X_all[tr_idx].copy(), X_all[va_idx].copy()
    ytr_f, yva_f = y_full[tr_idx], y_full[va_idx]
    med_f = np.nanmedian(Xtr_f[:, inc_idx])
    for arr in (Xtr_f, Xva_f):
        arr[:, inc_idx] = np.where(np.isnan(arr[:, inc_idx]), med_f, arr[:, inc_idx])
    sc_f = StandardScaler().fit(Xtr_f)
    Xtr_fs, Xva_fs = sc_f.transform(Xtr_f), sc_f.transform(Xva_f)
    train_ds_f = TabularDataset(Xtr_fs, ytr_f)
    train_loader_f = DataLoader(train_ds_f, batch_size=64, shuffle=True,
                                 generator=torch.Generator().manual_seed(SEED + fold))
    torch.manual_seed(SEED + fold)
    m = ClfDeepMLP(in_dim=len(FEATURE_COLS)).to(device)
    pos_f, neg_f = (ytr_f == 1).sum(), (ytr_f == 0).sum()
    crit_f = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([neg_f / pos_f]))
    opt_f = torch.optim.Adam(m.parameters(), lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-4)
    sch_f = torch.optim.lr_scheduler.CosineAnnealingLR(opt_f, T_max=100)
    for epoch in range(100):
        m.train()
        for xb, yb in train_loader_f:
            xb, yb = xb.to(device), yb.to(device)
            opt_f.zero_grad()
            crit_f(m(xb), yb).backward()
            opt_f.step()
        sch_f.step()
    m.eval()
    with torch.no_grad():
        Xv = torch.from_numpy(Xva_fs.astype(np.float32)).to(device)
        yv = torch.from_numpy(yva_f.astype(np.float32)).view(-1, 1).to(device)
        logits = m(Xv)
        loss_v = bce(logits, yv).item()
        prob_v = torch.sigmoid(logits).cpu().numpy().ravel()
    pred_v = (prob_v >= 0.5).astype(int)
    fold_rows.append({
        "fold": fold,
        "test_loss_BCE": round(float(loss_v), 4),
        "accuracy": round(float(accuracy_score(yva_f, pred_v)), 4),
        "precision": round(float(precision_score(yva_f, pred_v, zero_division=0)), 4),
        "recall": round(float(recall_score(yva_f, pred_v, zero_division=0)), 4),
        "f1": round(float(f1_score(yva_f, pred_v, zero_division=0)), 4),
        "roc_auc": round(float(roc_auc_score(yva_f, prob_v)), 4),
    })
    print(f"fold {fold}: F1={fold_rows[-1]['f1']:.4f} BCE={fold_rows[-1]['test_loss_BCE']:.4f}")

kfold_df = pd.DataFrame(fold_rows)
summary = {k: {"mean": round(float(kfold_df[k].mean()), 4), "std": round(float(kfold_df[k].std()), 4)}
           for k in ("test_loss_BCE", "accuracy", "precision", "recall", "f1", "roc_auc")}
print(kfold_df.to_string(index=False))
print(json.dumps(summary, indent=2))
with open("weights_V1_3/classification_pytorch_kfold_metrics.json", "w") as f:
    json.dump({"k": K, "folds": fold_rows, "summary": summary}, f, indent=2)

fold 1: F1=0.5749 BCE=0.3456


fold 2: F1=0.5455 BCE=0.4076


fold 3: F1=0.5258 BCE=0.4387


fold 4: F1=0.5436 BCE=0.4229


fold 5: F1=0.5641 BCE=0.3801
 fold  test_loss_BCE  accuracy  precision  recall     f1  roc_auc
    1         0.3456    0.8415     0.4800  0.7164 0.5749   0.8949
    2         0.4076    0.8210     0.4404  0.7164 0.5455   0.8662
    3         0.4387    0.7942     0.4016  0.7612 0.5258   0.8732
    4         0.4229    0.8009     0.4141  0.7910 0.5436   0.8720
    5         0.3801    0.8098     0.4264  0.8333 0.5641   0.9037
{
  "test_loss_BCE": {
    "mean": 0.399,
    "std": 0.0368
  },
  "accuracy": {
    "mean": 0.8135,
    "std": 0.0186
  },
  "precision": {
    "mean": 0.4325,
    "std": 0.0302
  },
  "recall": {
    "mean": 0.7637,
    "std": 0.0502
  },
  "f1": {
    "mean": 0.5508,
    "std": 0.0191
  },
  "roc_auc": {
    "mean": 0.882,
    "std": 0.0163
  }
}
